# Bayesian gene expression error model
In this notebook, we will define and assess a model to compute error bars for gene expression measurements. We will run simulation-based calibration (SBC) to assess how well the model does on simulated data.

# Model description

We will start by defining a Bayesian model. The model assumes the dataset contains unnormalized mRNA counts that have had the ERCC spike-in's added before library preparation. We define the following for each condition:

* $G$: the number of genes
* $K$ the number of spike-in species
* $x^{mRNA}$: a vector of length $G$ representing the unnormalized mRNA counts for each gene
* $X^{mRNA}$: the total number of mRNA counts for each condition (sum of $x^{mRNA}$)
* $\alpha$: a vector of lengh $G$ representing the relative proportions of each gene
* $b^x$: a vector of length $G$ representing the bias during library prep for the genes
* $s^{spike}$: a vector of length $K$ representing the input spike-in species values
* $s^{seq}$: a vector of length $K$ representing the spike-in species counts after sequencing
* $b^s$: a vector of length $K$ representing the bias during library prep for the spike-in species

Now, let's discuss the distributions.

In an idealized setting, $x^{mRNA}$ comes from a Multinomial distribution with probabilities $\alpha$ and number of trials $X^{mRNA}$. 

Based on the multiplicative nature of library prep, we can make a conjecture that $b_x$ and $b_s$ follow lognormal distributions with the same distribution parameters - $\mu_b$ and $\sigma_b$.

For the likelihoods, we assume the data follows a Poisson distribution - due to shot noise during sequencing.

Thus, we define the priors as:
* $x^{mRNA} \sim \text{ Multinomial}(n = X^{mRNA}, p = \alpha) $
* $X^{mRNA} \sim \text{LogNormal}(mu = np.log(200,000), sigma = 1)$
* $\alpha \sim \text{Dirichlet}(1_G)$
* $b^x \sim \text{LogNormal}(mu = \mu_b, sigma = \sigma_b)$
* $b^s \sim \text{LogNormal}(mu = \mu_b, sigma = \sigma_b)$
* $\mu_b \sim \text{Normal}(mu = 0, sigma = 1)$
* $\sigma_b \sim \text{HalfNormal}(sigma = 0.5)$

And the likelihoods:
* $s^{seq} \sim \text{Poisson}(b_s * s^{spike})$
* $x^{seq} \sim \text{Poisson}(b_x * x^{mRNA})$

Written all together in a Bayes formulation:
$$ \underset{\textcolor{purple}{\text{posterior}}}{\pi \left( \underline{\alpha}, \underline{b^{(x)}},\underline{b^{(s)}}, \mu_b, \sigma_b, \underline{x^{mRNA}}, X^{mRNA} | \underline{x^{seq}}, \underline{s^{seq}}, \underline{s^{spike}} \right)} \propto$$

$$\underset{\textcolor{purple}{\text{spike-in likelihood}}}{\pi \left(\underline{s^{spike}}, \underline{s^{seq}} | \underline{b^{(s)}} \right)}    \underset{\textcolor{purple}{\text{priors}}}{\pi \left(\underline{b^{(s)}} | \mu_b, \sigma_b \right)          \pi \left(\mu_b \right) \pi \left(\sigma_b \right)}\times$$

$$\underset{\textcolor{purple}{\text{txtome likelihood}}}{\pi \left(\underline{x^{seq}} | \underline{x^{mRNA}}, \underline{b^{(x)}} \right)}    \underset{\textcolor{purple}{\text{priors}}}{\pi \left(\underline{b^{(x)}} | \mu_b, \sigma_b \right)          }\times$$

$$ \underset{\textcolor{purple}{\text{idealized likelihood}}}{\pi \left( \underline{x^{mRNA}} | X^{mRNA}, \underline{\alpha}\right)}      \underset{\textcolor{purple}{\text{priors}}}{\pi \left(X^{mRNA} \right) \pi \left(\underline{\alpha} \right)}$$

# Imports

Must have numpyro, jax, bokeh, and iqplot installed on your computer. All these can be installed with `pip`.

In [6]:
import run_sbc
import plot_sbc

import numpy as np

import bokeh.io
bokeh.io.output_notebook()

Loading BokehJS ...

# Run SBC

For simulation-based calibration, we will first draw data from the parameters we have defined above, pass that data into the model, and take key measurements from the posterior (such as standard deviations, means, and ranks). To get a close look at how it's done, take a look at the `run_sbc()` function within `run_sbc.py`.

**Define parameters**

`num_iters` represents the number of SBC runs.

`num_genes` is the number of genes.

`s_spike` is the numpy array representing the spike-in inputs

`mu_b_sigma` is the sigma value for $mu_b$

`sigma_b_sigma` is the sigma value for $sigma_b$

`X_mrna_mu` is the mu value for $X^{mRNA}$

`X_mrna_sigma` is the sigma value for $X^{mRNA}$

`display_outputs` is a boolean variable. When `True`, the progress of warmup and steps of MCMC will be shown.


**The outputs**

`true_param_dict`: (dict) give the true values for the parameters from simulation

`mean_param_dict`: (dict) store the mean values of parameters from the posterior

`rank_param_dict`: (dict) stores the ranks of the parameters

`sd_param_dict`: (dict) stores the standard deviation values of parameters from the posterior

In [2]:
num_iters = 2
num_genes = 10

s_spike = np.array([2, 20, 200, 2000, 20000]) # approx. spike-in input values
mu_b_sigma = 1
sigma_b_sigma = 0.5
X_mrna_mu = np.log(200000) 
X_mrna_sigma = 1

display_outputs = False

**Run SBC**

In [3]:
true_param_dict, mean_param_dict, rank_param_dict, sd_param_dict = run_sbc.run_SBC(num_iters, G = num_genes,
                                                                                   s_spike = s_spike,
                                                                                   mu_b_sigma = mu_b_sigma,
                                                                                   sigma_b_sigma = sigma_b_sigma,
                                                                                   X_mrna_mu = X_mrna_mu,
                                                                                   X_mrna_sigma = X_mrna_sigma,
                                                                                  display_outputs = False)

100%|█████████████████████████████████████████████| 2/2 [00:10<00:00,  5.46s/it]


# Plot

We will take a look at the rank histograms, the rank ECDfs, and the scatter plots. We expect the histograms to be uniform or, equilavently, the ECDFs to be diagonal if the model is well calibrated. The 95% intervals computed from the K-S distribution are shown on the ECDFs. The scatter plots show the posterior mean vs true value for each parameter.

These functions can be found in `plot_sbc.py`.

**Relevant arguments**

`bins` in `plot_rank_histograms()` defines the number of bins to make the histogram with.

`every`: Save every nth plot for vector-valued parameters. For example, `every=5` will plot every 5th component of a vector parameter. This is especially useful when doing runs with a high number of genes.

In [7]:
plot_sbc.plot_rank_histograms(rank_param_dict, bins = 25, every = 1)

In [8]:
plot_sbc.plot_rank_ecdf(rank_param_dict, every = 1)

In [9]:
plot_sbc.plot_scatter(true_param_dict, mean_param_dict, every = 1)

In [10]:
%load_ext watermark
%watermark -v -p numpy,bokeh,numpyro,jax,iqplot,tqdm,jupyterlab

Python implementation: CPython
Python version       : 3.12.7
IPython version      : 8.27.0

numpy     : 1.26.4
bokeh     : 3.6.2
numpyro   : 0.18.0
jax       : 0.6.2
iqplot    : 0.3.7
tqdm      : 4.66.5
jupyterlab: 4.3.3

